In [1]:
import numpy as np

from odyn import Database
from analysis.seg_10x.gui import launch
from analysis.seg_10x.watershed import GLOM_10X_DEFAULTS, scale_params

[INFO] Import Database and Group classes for db UI and help functions!
[INFO] You can use 'from odyn import Database, Group' to import them.
[INFO] Hovering over 'Database' and 'Group' will give you some tips.


In [2]:
PROJECT = "PA_K99"
MAIN_FOLDER = "/Users/vinicius/GitHub/Workspace/odyn-pa-k99"
GROUP_ID = 53

db = Database(MAIN_FOLDER, project=PROJECT)
group = db.groups[GROUP_ID]

[INFO] Connected to the database at: '/Users/vinicius/GitHub/Workspace/odyn-pa-k99/.odyn/projects/PA_K99.db'


In [3]:
db.group_experiments.loc[53]

exp_id                                         46
exp_name                       20251024_sid237_e1
exp_type                                     loop
exp_start              2025-10-24 12:38:33.620000
mouse_id                                   sid237
height_px                                     600
width_px                                      476
height_um                                  1200.0
width_um                                    952.0
frame_count                                   301
frame_rate                              25.074918
laser_power_920                                80
laser_power_1040                                0
loop_acq_interval_s                          10.0
added_to_db_at                2026-08-22 20:58:23
Name: 53, dtype: object

In [ ]:
programs = group.programs[group.programs.program_type.isin(["fine 1", "fine 2", "coarse 1", "coarse 2"])]
trials = group.trials[group.trials.program_id.isin(programs.index)]
trials.groupby(["odor_id", "program_id", "outcome"]).agg(count=("acq_id", "count"))

In [4]:
# images_raw = np.load(f"{MAIN_FOLDER}/outputs/correlation_images_10721429.npz")
images_raw = np.load("/Users/vinicius/correlation_images_10739447.npz")

images = {}
images_max = {}
counts = {}

for key, image in images_raw.items():
    program_id, odor_id, outcome = key.split(".")

    if odor_id in images:
        images[odor_id] = image + images[odor_id]
        counts[odor_id] += 1

    else:
        images[odor_id] = image
        counts[odor_id] = 1

for odor_id in images:
    images[odor_id] /= counts[odor_id]


In [ ]:
exp = group.experiments.iloc[0]
um_per_px = float(exp["width_um"]) / int(exp["width_px"])

gui = launch(
    images,
    save_path=f"{MAIN_FOLDER}/masks_group_6.npz",
    params=scale_params(GLOM_10X_DEFAULTS, to_um_per_px=um_per_px)
)

Loading BokehJS ...

ERROR:bokeh.server.protocol_handler:error handling message
 message: Message 'PATCH-DOC' content: {'events': [{'kind': 'ModelChanged', 'model': {'id': 'p1081'}, 'attr': 'value_throttled', 'new': 47}]} 
 error: ValueError('min_diameter_px (47.0) must be below max_diameter_px (40.0).')
Traceback (most recent call last):
  File "/Users/vinicius/miniforge3/envs/caiman/lib/python3.13/site-packages/bokeh/server/protocol_handler.py", line 94, in handle
    work = await handler(message, connection)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vinicius/miniforge3/envs/caiman/lib/python3.13/site-packages/bokeh/server/session.py", line 94, in _needs_document_lock_wrapper
    result = func(self, *args, **kwargs)
  File "/Users/vinicius/miniforge3/envs/caiman/lib/python3.13/site-packages/bokeh/server/session.py", line 286, in _handle_patch
    message.apply_to_document(self.document, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vinicius/miniforge3/envs/cai